# 07. Gradient updates and parameter schedules

![One effective parameter update, from microbatch losses through clipping and AdamW to new parameters](../images/07_gradient_updates_and_schedules.svg)

The lecture followed one loss value into one parameter update. This notebook zooms out to the whole run and shows how a fixed training budget turns into a scientific contract for the iso-catalog phase-allocation study.

Three things get pinned down here, in order: how many optimizer updates a clip budget buys, how the exposure tier is chosen before anyone sees an outcome, and what a checkpoint must record before a resume is allowed to proceed.

Read the [lecture](../lectures/07_gradient_updates_and_schedules.md) first if you have not, and return to the [tutorial index](../README.md) at any point.

**Learning goals:** turn a clip budget, a batch size, and an accumulation factor into an exact number of updates; verify that two conditions share one exposure tier; choose that tier while still blind to outcomes; and store enough provenance to reject an incompatible resume.

The setup cell below fixes a seed so every run of this notebook produces the same numbers. Nothing here trains a model. The point is the bookkeeping that makes trained models comparable.

In [ ]:
from dataclasses import dataclass
import math
import numpy as np

SEED = 17
rng = np.random.default_rng(SEED)
assert rng.integers(1, 10) >= 1


## Fixed clips imply fixed updates

A comparison is about allocation only when every model receives the same clip exposure and the same effective batch size. Otherwise a difference in results could just be a difference in optimization.

The effective batch is the product of three numbers: clips per device, number of devices, and accumulation steps. Below that is `32 * 8 * 2 = 512` clips per update. A budget of 4,096,000 clips therefore buys 8,000 updates, and 8,192,000 clips buys 16,000.

The function refuses budgets that do not divide evenly. A remainder would mean a partial final batch, and a partial batch is a silent difference between two runs that were supposed to match.

In [ ]:
def completed_updates(clips, per_device_batch, devices, accumulation):
    effective_batch = per_device_batch * devices * accumulation
    if clips % effective_batch:
        raise ValueError('clip exposure must divide the effective batch')
    return clips // effective_batch, effective_batch

for clips in (4_096_000, 8_192_000):
    updates, batch = completed_updates(clips, 32, 8, 2)
    print(clips, updates, batch)
    assert updates * batch == clips


## Choose the tier before outcomes

The previous cell assumed a clip budget. This cell decides it. A storage probe runs eight training-like jobs at once and measures the sustained rate of each one, so `rates` holds eight numbers in examples per second per GPU.

The probe may select the full or the half exposure tier. It may never select a different tier for a favorable allocation, which is why the decision happens here, before any model is trained and long before any result is read.

The check below is a simplified stand-in for the frozen systems rule in the lecture. It requires every one of the eight jobs to clear a floor rate and requires the fastest job to stay within 1.5 times the slowest, because a wide spread means the shared filesystem, not the GPUs, is setting the pace.

In [ ]:
def storage_stable(rates):
    rates = np.asarray(rates, dtype=float)
    if rates.shape != (8,):
        raise ValueError('rates.shape != (8,)')
    return rates.min() >= 30.0 and rates.max() / rates.min() <= 1.5

rates = np.array([62, 60, 61, 59, 63, 60, 62, 61], dtype=float)
assert storage_stable(rates)
EXPOSURE = 8_192_000 if storage_stable(rates) else 4_096_000
assert EXPOSURE in {4_096_000, 8_192_000}


## Resume is a protocol check

A checkpoint is not resumable just because its weights load without an exception. It has to belong to the same allocation, the same phase catalog, the same exposure, and the same random streams, or the continued run is a new experiment wearing an old run's optimizer state.

`REQUIRED_RESUME_FIELDS` names that contract exactly: the two digests, the allocation with its sequence count, origins per sequence, and nominal catalog size, the origin policy, the exposure and batch size, the completed update count, both seeds, and the four stream versions. The set is compared with `==`, so a missing field and an unexpected extra field both fail.

`validate_resume` then checks internal consistency, requiring `nominal_catalog_size` to equal unique sequences times origins per sequence, before comparing the saved row against the requested one. The last block changes only the phase-catalog digest and asserts that the load refuses it. Failing closed on one changed field is the behavior you want, because the alternative is a warning nobody reads.

In [ ]:
REQUIRED_RESUME_FIELDS = {
    'manifest_digest', 'phase_catalog_digest', 'allocation',
    'unique_sequences', 'origins_per_sequence', 'nominal_catalog_size',
    'origin_policy', 'planned_exposure', 'effective_batch',
    'completed_updates', 'optimization_seed', 'replicate_seed',
    'sequence_stream_version', 'phase_stream_version',
    'spatial_stream_version', 'mask_stream_version',
}
ALLOCATIONS = {'breadth', 'balanced', 'phase_depth', 'nearby_jitter'}
ORIGIN_POLICIES = {'base_phase', 'phase_separated', 'nearby_jitter'}

row = {
    'manifest_digest': 'synthetic-manifest-v2',
    'phase_catalog_digest': 'synthetic-phase-v1',
    'allocation': 'phase_depth', 'unique_sequences': 62_500,
    'origins_per_sequence': 4, 'nominal_catalog_size': 250_000,
    'origin_policy': 'phase_separated', 'planned_exposure': EXPOSURE,
    'effective_batch': 512, 'completed_updates': EXPOSURE // 512,
    'optimization_seed': 13, 'replicate_seed': 17,
    'sequence_stream_version': 'sequence-v2',
    'phase_stream_version': 'phase-v1',
    'spatial_stream_version': 'spatial-v1', 'mask_stream_version': 'mask-v1',
}

def validate_resume(saved, expected):
    if set(saved) != REQUIRED_RESUME_FIELDS:
        raise ValueError('resume metadata has missing or unknown fields')
    if saved['allocation'] not in ALLOCATIONS or saved['origin_policy'] not in ORIGIN_POLICIES:
        raise ValueError('unknown allocation or origin policy')
    if saved['nominal_catalog_size'] != saved['unique_sequences'] * saved['origins_per_sequence']:
        raise ValueError('nominal catalog cardinality is inconsistent')
    if saved != expected:
        raise ValueError('resume metadata does not match the frozen row')

validate_resume(row, dict(row))
try:
    changed = dict(row); changed['phase_catalog_digest'] = 'other'
    validate_resume(changed, row)
except ValueError:
    pass
else:
    raise AssertionError('phase-catalog mismatch must fail closed')


**Takeaway:** fixed exposure is necessary and not sufficient. Comparability also needs a fixed allocation definition, a fixed phase catalog, pinned stream versions, one checkpoint rule, and a resume that fails closed. Exposure equalizes how much optimization each model receives. The rest of the contract equalizes everything else.

Lesson 08 picks up the one thing this notebook deliberately left abstract: what the clips actually are, and what makes two of them different.

Previous: [06. Representation collapse](06_representation_collapse.ipynb) · Next: [08. Group-aware sampling](08_group_aware_sampling.ipynb)